# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

This section lists all record sets (`@id`) in the dataset along with their associated fields and columns.

**Note:** By convention, all entities are referenced by their `@id` from the Croissant schema.

In [ ]:
# List all record sets in the dataset
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s) in the dataset.\n")

for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for f in fields:
        print(f"    {f['@id']} (label: {f.get('name', '')})")
        columns = f.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        if columns:
            print("      Columns:")
            for c in columns:
                print(f"        {c['@id']} (label: {c.get('name', '')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

The example below demonstrates loading all record sets into pandas DataFrames using their `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Extract records for each record set
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records for RecordSet: {record_set_id}")
            print(f"Fields/columns: {list(dataframes[record_set_id].columns)}\n")
        else:
            print(f"No records found for RecordSet: {record_set_id}\n")
    except Exception as e:
        print(f"Could not load records for RecordSet {record_set_id}: {e}\n")

# Example: show the head of the first DataFrame (if any exist)
if dataframes:
    example_rs_id = next(iter(dataframes.keys()))
    print(f"Example data from RecordSet {example_rs_id}:")
    display(dataframes[example_rs_id].head())
else:
    print("No DataFrames created.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, choose a numeric field and perform simple filtering and normalization. Adjust `numeric_field_id` and `group_field_id` as available from above.

In [ ]:
# EDA Example using a record set and numeric field
# Please replace these with actual @id from the Data Overview output if needed

# Pick the first record set and try to find a numeric field
if dataframes:
    rs_id = next(iter(dataframes.keys()))
    df = dataframes[rs_id]
    print(f"Exploring RecordSet {rs_id} (shape={df.shape})")
    # Attempt to auto-select a numeric field
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field: {numeric_field}")
    else:
        print("No numeric fields found for EDA.")
        numeric_field = None

    if numeric_field:
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to group by a categorical field (if one exists)
        categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        # Remove empty/uninformative columns
        categorical_cols = [c for c in categorical_cols if df[c].nunique() > 1]
        if categorical_cols:
            group_field = categorical_cols[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.reset_index().head())
        else:
            print("No suitable categorical/grouping field found.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below, we produce a histogram or box plot if a numeric field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field from previous step, if available
if dataframes and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field} in {rs_id}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If normalization was performed, also show boxplot
    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=filtered_df[f"{numeric_field}_normalized"])
        plt.title(f"Boxplot of Normalized {numeric_field}")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Explored the Croissant-metadata for the dataset and discovered its record sets, fields, and columns using their `@id`.
- Loaded records into pandas DataFrames for simple EDA and visualization.
- Filtered and normalized a numeric field (if available), and grouped by a key attribute for further analysis.
- Visualized field distributions to facilitate interpretation of the ordered logistic regression outputs.

Further analysis can include exploring relationships between additional fields, modeling, or advanced visualization as appropriate for downstream ML or statistical workflows.